In [1]:
import os
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

In [6]:
persitent_directory = "vectordb"
pdf_directory = "pdf_files"

In [3]:
def process_all_pdfs():
    pdf_files = [f for f in os.listdir(PDF_DIR) if f.endswith('.pdf')]
    all_chunks=[]
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    for file in pdf_files:
        path = os.path.join(PDF_DIR, file)
        loader = PyPDFLoader(path)
        docs=loader.load()
        for doc in docs:
            doc.metadata['source']=path
        chunks=splitter.split_documents(docs)
        all_chunks.extend(chunks)
    vectordb=Chroma.from_documents(
        documents=all_chunks,
        embedding=OpenAIEmbeddings(),
        persist_directory=persitent_directory
    )
    vectordb.persist()



In [7]:
def load_vector_store():
    vectordb = Chroma(
        persist_directory=persitent_directory,
        embedding_function=OpenAIEmbeddings()
    )
    return vectordb
def get_qa_chain(vectordb):
    retriever=vectordb.as_retriever(search_kwargs={"k": 3})
    chain =RetrievalQA.from_chain_type(
        llm=ChatOpenAI(temperature=0),
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True
    )
    return chain
